# Week 4 Exercise: Python → C++ Code Converter

Using frontier LLMs to translate Python code into optimised, high-performance C++ code.

This notebook:
- Converts a set of Python functions to C++ using **GPT-4o**, **Claude Sonnet**, and **Gemini**
- Streams the generated C++ with syntax highlighting
- Optionally compiles and benchmarks the output on your local machine
- Prints a side-by-side timing comparison for each model

**Author:** Uchebuzz (uche.buzugbe@analyticsintelligence.com)

In [ ]:
import os
import time
import subprocess
import tempfile
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [ ]:
# ── Models ────────────────────────────────────────────────────────────────────
OPENAI_MODEL   = "gpt-4o"
CLAUDE_MODEL   = "claude-sonnet-4-5-20250929"
GEMINI_MODEL   = "gemini-2.0-flash"

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are an expert C++ engineer. When given Python source code, produce the equivalent
C++ (C++17) that:
  • compiles cleanly with g++ -std=c++17 -O2
  • preserves the exact semantics of the Python version
  • uses idiomatic modern C++ (auto, range-for, STL algorithms where appropriate)
  • includes a main() that runs the function and prints its result to stdout

Return ONLY the C++ source code — no markdown fences, no explanation.
"""

In [ ]:
load_dotenv(override=True)

openai_key     = os.getenv("OPENAI_API_KEY")
anthropic_key  = os.getenv("ANTHROPIC_API_KEY")
google_key     = os.getenv("GOOGLE_API_KEY")

openai_client  = OpenAI(api_key=openai_key)
claude_client  = OpenAI(
    api_key  = anthropic_key,
    base_url = "https://api.anthropic.com/v1/"
)
gemini_client  = OpenAI(
    api_key  = google_key,
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
)

for label, key in [("OpenAI", openai_key), ("Anthropic", anthropic_key), ("Google", google_key)]:
    status = f"begins {key[:6]}" if key else "NOT SET"
    print(f"{label}: {status}")

## Python functions to convert

Three classic compute-heavy examples that benefit from C++ performance:

In [ ]:
# ── Example 1: Sieve of Eratosthenes ─────────────────────────────────────────
PYTHON_SIEVE = """\
def sieve_of_eratosthenes(limit: int) -> list[int]:
    """Return all prime numbers up to `limit` (inclusive)."""
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, limit + 1, i):
                is_prime[j] = False
    return [i for i, p in enumerate(is_prime) if p]

primes = sieve_of_eratosthenes(1_000_000)
print(f"Found {len(primes)} primes up to 1,000,000")
print(f"Largest prime: {primes[-1]}")
"""

# ── Example 2: Matrix multiply ────────────────────────────────────────────────
PYTHON_MATMUL = """\
def matmul(A: list[list[float]], B: list[list[float]]) -> list[list[float]]:
    """Naive O(n^3) matrix multiplication."""
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            for j in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C

N = 200
A = [[float(i + j) for j in range(N)] for i in range(N)]
B = [[float(i * j + 1) for j in range(N)] for i in range(N)]
C = matmul(A, B)
print(f"C[0][0] = {C[0][0]:.4f}")
print(f"C[{N-1}][{N-1}] = {C[N-1][N-1]:.4f}")
"""

PYTHON_EXAMPLES = {
    "Sieve of Eratosthenes": PYTHON_SIEVE,
    "Matrix Multiply (200×200)": PYTHON_MATMUL,
}

In [ ]:
def stream_cpp(client: OpenAI, model: str, python_code: str, label: str) -> str:
    """Stream C++ output from the given model and return the full text."""
    display(Markdown(f"### `{label}` → `{model}`"))
    display_handle = display(Markdown("*Generating…*"), display_id=True)

    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Convert this Python code to C++:\n\n{python_code}"},
        ],
        stream=True,
    )

    chunks = []
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        chunks.append(delta)
        update_display(
            Markdown(f"```cpp\n{''.join(chunks)}\n```"),
            display_id=display_handle.display_id,
        )

    return "".join(chunks)

In [ ]:
def compile_and_run(cpp_code: str) -> tuple[bool, str]:
    """
    Write cpp_code to a temp file, compile with g++, run it.
    Returns (success, output_or_error).
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        src  = Path(tmpdir) / "main.cpp"
        exe  = Path(tmpdir) / "main"
        src.write_text(cpp_code, encoding="utf-8")

        compile_result = subprocess.run(
            ["g++", "-std=c++17", "-O2", "-o", str(exe), str(src)],
            capture_output=True, text=True
        )
        if compile_result.returncode != 0:
            return False, compile_result.stderr

        run_result = subprocess.run(
            [str(exe)], capture_output=True, text=True, timeout=30
        )
        return True, run_result.stdout

## Generate C++ from each model

In [ ]:
MODELS = [
    (openai_client,  OPENAI_MODEL,  "OpenAI"),
    (claude_client,  CLAUDE_MODEL,  "Claude"),
    (gemini_client,  GEMINI_MODEL,  "Gemini"),
]

results: dict[str, dict[str, str]] = {}  # results[example][model_label] = cpp_code

for example_name, python_code in PYTHON_EXAMPLES.items():
    display(Markdown(f"---\n## {example_name}\n\n**Python source:**\n```python\n{python_code}\n```"))
    results[example_name] = {}

    for client, model, label in MODELS:
        try:
            t0  = time.perf_counter()
            cpp = stream_cpp(client, model, python_code, label)
            elapsed = time.perf_counter() - t0
            results[example_name][label] = cpp
            display(Markdown(f"*Generated in {elapsed:.1f}s*"))
        except Exception as exc:
            display(Markdown(f"**Error from {label}:** `{exc}`"))

## Compile & run (optional)

Requires `g++` installed locally. On Windows use [MSYS2](https://www.msys2.org/) or WSL.

In [ ]:
for example_name, model_outputs in results.items():
    display(Markdown(f"### {example_name}"))
    rows = ["| Model | Compiled? | Output |",
            "|-------|-----------|--------|"]

    for label, cpp_code in model_outputs.items():
        try:
            ok, output = compile_and_run(cpp_code)
            status = "✅" if ok else "❌"
            snippet = output.strip().replace("\n", " ")[:120] if ok else output.strip()[:120]
        except FileNotFoundError:
            status  = "⚠️"
            snippet = "`g++` not found — paste code into [godbolt.org](https://godbolt.org)"
        except Exception as exc:
            status  = "❌"
            snippet = str(exc)[:120]

        rows.append(f"| {label} | {status} | {snippet} |")

    display(Markdown("\n".join(rows)))

## Reflections

All three frontier models produced valid, idiomatic C++ from straightforward Python. Key observations:

- **GPT-4o** tends to produce the most concise output with clean use of STL containers.
- **Claude Sonnet** adds helpful inline comments explaining algorithmic choices, which aids readability.
- **Gemini Flash** is the fastest to respond and still produces compilable code, though occasionally omits `#include` headers.

For production use I'd recommend passing the generated C++ back to the model with any compiler errors for a self-correcting loop.